# Rapport — Engine de traitement et d'analyse de flux de données sécurisé

**Niveau :** ING3  
**Langage :** Python 3.10+  
**Forme :** application en ligne de commande, package `secure_log_engine`, point d'entrée `main.py`  
**Rôle de ce notebook :** rapport de projet. Il explique l'architecture, justifie les choix imposés par le cahier des charges, et exécute le moteur sur le jeu de logs fourni.

Le notebook n'ajoute aucune dépendance. Tout ce qu'il importe appartient à la bibliothèque standard, comme le reste du projet (`requirements.txt` ne fige donc aucune bibliothèque tierce). Il doit être ouvert avec le répertoire du projet comme dossier de travail, pour que `import secure_log_engine` et `import main` trouvent le package.

## Sommaire

1. Problème et contraintes
2. Barème et emplacement dans le code
3. Architecture des fichiers
4. Environnement
5. Jeu de données
6. Modèle de données
7. Module 1 — ingestion par générateurs
8. Module 1 — parsing, normalisation, validation
9. Module 2 — pipeline fonctionnel
10. Module 3 — ensembles, compteurs, force brute
11. Module 4 — moteur de règles
12. Module 4 — audit par introspection
13. Pipeline complet de `main.py`
14. Tests
15. Limites connues
16. Conclusion


## 1. Problème et contraintes

Le sujet demande une plateforme capable d'ingérer, filtrer, transformer et analyser des journaux système ou réseau volumineux, sans placer le fichier entier en mémoire vive.

Quatre idées organisent le code.

**Le flux reste un itérateur.** Un fichier de logs se lit ligne par ligne. Le producteur (`yield`) et le consommateur (boucle `for`) avancent ensemble. On ne construit pas la liste de toutes les lignes avant de commencer l'analyse.

**Chaque ligne devient une valeur typée.** Le texte brut est trop fragile pour des règles de sécurité. Le parsing produit un `LogRecord` : date, niveau, IP, utilisateur, événement, payload, et éventuellement un tuple de connexion immuable.

**L'analyse est séparée de la lecture.** Compter, comparer une blacklist, détecter une force brute : ces opérations consomment des `LogRecord`, elles ne rouvrent pas les fichiers.

**Les règles sont des fonctions.** Le module 4 ne code pas une liste figée de contrôles dans `main.py`. Il reçoit des fonctions et un dictionnaire de paramètres. Ajouter une règle, c'est passer une fonction de plus.

Contraintes de rendu respectées par le dépôt :

- environnement isolable par `venv` ;
- `requirements.txt` présent (bibliothèque standard seule, donc fichier sans paquet à installer) ;
- package Python importable, pas un seul script ;
- `main.py` comme point d'entrée de démonstration ;
- tests `unittest` ;
- ce notebook comme rapport, les notebooks n'étant prévus par le sujet que pour la démonstration finale.


## 2. Barème et emplacement dans le code

| Critère du sujet | Où c'est fait | Idée directrice |
| --- | --- | --- |
| Environnement et livrables (2 pts) | `requirements.txt`, `pyproject.toml`, `README.md`, package `.py`, ce notebook | Aucune bibliothèque hors standard |
| Générateurs et mémoire (4 pts) | `secure_log_engine/ingestion.py` | `yield` pour un fichier, `yield from` pour plusieurs fichiers, `simulate_logs` pour un flux sans fichier |
| Structures (4 pts) | `models.py`, `parsing.py`, `analysis.py` | `tuple` de connexion, `set`, `frozenset`, `defaultdict`, `match` / `case` |
| Programmation fonctionnelle (4 pts) | `pipeline.py`, `rules.py` | `filter`, `map`, `lambda`, fonctions passées en argument |
| Moteur de règles et introspection (3 pts) | `rules.py`, `audit.py`, appel dans `main.py` | `apply_security_rules(*args, **kwargs)`, `globals()`, `locals()` |
| Qualité (3 pts) | tout le package | annotations de type, exceptions dédiées, `for` / `while` avec `break`, `continue` et `else` |

`demo_personne1.py` est une démonstration réduite à l'ingestion et au parsing. Le parcours complet, celui qui est noté comme pipeline, est `main.py`.


## 3. Architecture

```text
fichiers .log  ou  simulate_logs()
        |
        |  iter_file_lines / iter_sources     ingestion.py
        v
   lignes de texte  (générateur)
        |
        |  parse_log_line                     parsing.py
        |  normalize_record
        |  valid_record
        v
   LogRecord  (dataclass gelée)
        |
        +--> ips en erreur, defaultdict, blacklist     analysis.py
        |
        +--> detecter_force_brute
        |
        +--> apply_security_rules(*regles, **config)   rules.py
        |
        +--> auditer_contexte(locals())                audit.py
        v
   affichage CLI dans main.traiter_les_logs
```

`pipeline.py` expose la version fonctionnelle du nettoyage (`filter`, `map`, `lambda`, second `match`). `main.py` ne l'appelle pas : il réécrit la même séquence parse / normalise / valide dans `traiter_les_logs`, parce que cette fonction doit en même temps compter les rejets et alimenter l'analyse. Le module 2 reste testable et est exécuté plus bas dans ce rapport. Cette séparation est une limite, elle est décrite à la section 15.

Le package `secure_log_engine/__init__.py` réexporte les fonctions publiques. Un appelant peut donc écrire `from secure_log_engine import apply_security_rules` sans connaître le fichier.


In [1]:
from pathlib import Path
import sys

RACINE = Path.cwd().resolve()
if not (RACINE / "secure_log_engine" / "__init__.py").exists():
    raise SystemExit(
        "Le notebook doit être lancé depuis la racine du projet "
        "(le dossier qui contient main.py)."
    )
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

print(f"Racine du projet : {RACINE}")
print(f"Python           : {sys.version.split()[0]}")
print("Fichiers du moteur :")
for chemin in sorted((RACINE / "secure_log_engine").glob("*.py")):
    print(f"  - {chemin.relative_to(RACINE)}")


Racine du projet : C:\Users\theop\Downloads\Projet-Python-main 2\Projet-Python-main
Python           : 3.13.15
Fichiers du moteur :
  - secure_log_engine\__init__.py
  - secure_log_engine\analysis.py
  - secure_log_engine\audit.py
  - secure_log_engine\ingestion.py
  - secure_log_engine\models.py
  - secure_log_engine\parsing.py
  - secure_log_engine\pipeline.py
  - secure_log_engine\rules.py


## 4. Environnement

Le sujet interdit une bibliothèque qui ne serait pas figée dans le fichier d'environnement. Le moteur s'en tient à la bibliothèque standard : `pathlib`, `json`, `ast`, `ipaddress`, `dataclasses`, `collections`, `itertools`, `argparse`, `unittest`.

Conséquence pratique : `pip install -r requirements.txt` réinstalle un environnement vide de paquets tiers, et le programme tourne dès que Python 3.10 ou plus est disponible. Le structural pattern matching (`match` / `case`) et les unions de types `str | dict` exigent cette version. Ils ne sont pas réécrivables en Python 3.9 sans changer le code.

Création de l'environnement, rappelée dans le README :

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python main.py
python -m unittest discover tests
```


In [2]:
from pathlib import Path
print((RACINE / "requirements.txt").read_text(encoding="utf-8"))
print("--- pyproject.toml (extrait des métadonnées) ---")
texte = (RACINE / "pyproject.toml").read_text(encoding="utf-8")
print("\n".join(texte.splitlines()[:12]))


# ==============================================================================
# Fichier de dépendances - Environnement Virtuel (venv)
# Projet: Engine de traitement et d'analyse de flux de données sécurisé en temps réel
# ==============================================================================
#
# Le cœur du moteur (Module 1 : ingestion, modélisation et parsing streaming)
# s'appuie exclusivement sur la bibliothèque standard Python 3.10+ pour
# maximiser les performances et minimiser l'empreinte mémoire.
#
# Pour réinstaller l'environnement :
#   python -m venv .venv
#   source .venv/bin/activate       # (ou .venv\Scripts\Activate sous Windows)
#   pip install -r requirements.txt
#
# (Généré / synchronisé via pip freeze)


--- pyproject.toml (extrait des métadonnées) ---
[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "secure-log-engine"
version = "0.1.0"
description = "Engine de traitement et d'analyse de flux de donnée

## 5. Jeu de données

Deux fichiers tiennent lieu de flux réaliste, volontairement petit pour que chaque décision du moteur soit vérifiable à la main.

`data/sample_1.log` mélange déjà les trois formats attendus, plus les cas qui doivent être rejetés :

- objets JSON, avec un niveau écrit en minuscules (`info`, `warning`) pour forcer la normalisation ;
- une ligne à champs séparés par `|`, dont le payload est lui-même un JSON d'erreur ;
- une ligne vide ;
- une ligne qui n'a aucune forme reconnue (`LIGNE_MAL_FORMEE`) ;
- une ligne syntaxiquement correcte mais dont l'IP n'existe pas (`pas-une-ip`).

`data/sample_2.log` ajoute un tuple Python (connexion SSH), des échecs de login répétés depuis l'IP blacklistée `203.0.113.66`, et un `DISK_ERROR` au format dictionnaire.

La blacklist du programme est le `frozenset` `{"203.0.113.66", "198.51.100.99"}`. La seconde adresse n'apparaît dans aucun fichier. Elle sert à montrer qu'une différence d'ensembles n'est pas une intersection : une IP surveillée peut n'avoir produit aucun log.

Les adresses `192.0.2.0/24`, `198.51.100.0/24` et `203.0.113.0/24` sont des plages de documentation (RFC 5737). Le jeu de test ne contient donc aucune adresse opérationnelle.


In [3]:
from pathlib import Path

def montrer_fichier(nom):
    print("=" * 78)
    print(nom)
    print("=" * 78)
    lignes = (RACINE / "data" / nom).read_text(encoding="utf-8").splitlines()
    for numero, ligne in enumerate(lignes, start=1):
        visible = ligne if ligne.strip() else "∅ ligne vide"
        print(f"{numero:02d} | {visible}")
    print(f"({len(lignes)} lignes, affichage seul : le moteur, lui, n'utilise pas read().splitlines())")
    print()

montrer_fichier("sample_1.log")
montrer_fichier("sample_2.log")


sample_1.log
01 | {"timestamp":"2026-11-01T08:00:00Z","level":"info","ip":"192.0.2.1","user":"alice","event":"LOGIN_SUCCESS","payload":{"action":"login","result":"ok"},"connection":["192.0.2.1",443,"tcp"]}
02 | {"timestamp":"2026-11-01T08:01:00Z","level":"warning","ip":"192.0.2.1","user":"alice","event":"LOGIN_FAILED","payload":{"reason":"bad_password"}}
03 | {"timestamp":"2026-11-01T08:02:00Z","level":"WARNING","ip":"192.0.2.1","user":"alice","event":"LOGIN_FAILED","payload":"Mot de passe incorrect"}
04 | {"timestamp":"2026-11-01T08:03:00Z","level":"WARNING","ip":"192.0.2.1","user":"alice","event":"LOGIN_FAILED","payload":"Troisième tentative"}
05 | 2026-11-01T08:04:00Z|ERROR|203.0.113.66|bob|API_ERROR|{"level":"ERROR","code":500}
06 | {"timestamp":"2026-11-01T08:04:30Z","level":"CRITICAL","ip":"203.0.113.66","user":"bob","event":"FIREWALL_BLOCK","payload":{"detail":"Accès non autorisé"}}
07 | ∅ ligne vide
08 | LIGNE_MAL_FORMEE
09 | {"timestamp":"2026-11-01T08:05:00Z","level":"ERROR",

## 6. Modèle de données

Le fichier `models.py` ne traite rien. Il fixe la forme des valeurs qui circulent ensuite.

**`Connection`** est l'alias de `tuple[str, int, str]`, lu comme `(adresse IP, port, protocole)`. Un tuple ne peut pas être modifié après création. Une métadonnée de connexion (le port ne « devient » pas un autre port au milieu du pipeline) n'a pas besoin d'un objet mutable.

**`Payload`** est l'union `str | dict | tuple`. Le sujet demande explicitement ces trois formes : chaîne brute, dictionnaire, tuple de connexion. Le parsing refuse d'inventer un quatrième type : une liste JSON est convertie en tuple avant d'entrer dans le `LogRecord`.

**`LogRecord`** est une `dataclass` gelée (`frozen=True`). Affecter `log.level = "INFO"` lève `FrozenInstanceError`. La normalisation ne modifie donc pas l'objet reçu : `normalize_record` renvoie un nouvel enregistrement via `dataclasses.replace`.

**`Finding`** est le format unique des alertes. Le module 3 le remplit avec la date, l'IP et l'utilisateur. Le module 4 ajoute deux champs optionnels, `rule` et `detail`, qui ont une valeur par défaut vide. Les anciens appels `Finding(date, ip, user)` restent valides.

Pourquoi geler les enregistrements : une règle de sécurité qui corrigerait silencieusement un champ fausserait l'audit. Si la donnée doit changer, le changement est une fonction pure qui retourne une nouvelle valeur, ce que fait la normalisation.


In [4]:
import json
from dataclasses import FrozenInstanceError

from secure_log_engine.models import Connection, Finding, LogRecord
from secure_log_engine.parsing import normalize_record, parse_log_line

ligne = json.dumps({
    "timestamp": "2026-11-01T08:00:00Z",
    "level": " error ",
    "ip": " 192.0.2.1 ",
    "user": " Alice ",
    "event": " login_failed ",
    "payload": {"reason": "bad_password"},
    "connection": ["192.0.2.1", 443, "tcp"],
})
brut = parse_log_line(ligne)
propre = normalize_record(brut)

print("Type de la connexion :", type(propre.connection).__name__, propre.connection)
print("La connexion est bien un tuple Connection :", isinstance(propre.connection, tuple))
print("Niveau avant normalisation :", repr(brut.level))
print("Niveau après normalisation :", repr(propre.level))
print("Objet différent après replace :", brut is not propre)
print("Utilisateur nettoyé :", repr(propre.user), "| événement :", propre.event)

try:
    propre.connection = ("10.0.0.1", 80, "TCP")
except FrozenInstanceError as erreur:
    print("LogRecord gelé :", type(erreur).__name__)

connexion: Connection = propre.connection
try:
    connexion[1] = 22
except TypeError as erreur:
    print("Tuple immuable :", type(erreur).__name__)

alerte = Finding(propre.timestamp, propre.ip, propre.user)
print("Alerte du module 3, règle encore vide :", alerte)


Type de la connexion : tuple ('192.0.2.1', 443, 'TCP')
La connexion est bien un tuple Connection : True
Niveau avant normalisation : ' error '
Niveau après normalisation : 'ERROR'
Objet différent après replace : True
Utilisateur nettoyé : 'Alice' | événement : LOGIN_FAILED
LogRecord gelé : FrozenInstanceError
Tuple immuable : TypeError
Alerte du module 3, règle encore vide : Finding(timestamp=datetime.datetime(2026, 11, 1, 8, 0, tzinfo=datetime.timezone.utc), ip='192.0.2.1', user='Alice', rule='', detail='')


## 7. Module 1 — ingestion par générateurs

### Ce qu'un générateur change

`Path.read_text()` ou `list(fichier)` matérialise tout le contenu avant la première analyse. Sur un journal de production, cette liste double le coût : le texte est en RAM, puis les objets parsés le sont aussi.

`iter_file_lines` fait l'inverse. La fonction contient `yield`, donc l'appeler ne lit encore rien : elle renvoie un générateur. Le `with` qui ouvre le fichier est suspendu avec la fonction. Chaque `next`, ou chaque tour de `for`, reprend la fonction, lit une ligne, retire le saut de ligne, et se suspend de nouveau. Quand le consommateur s'arrête, le générateur est abandonné et le fichier se ferme avec lui.

`iter_sources` enchaîne plusieurs chemins. `yield from iter_file_lines(path)` délègue au générateur du fichier, sans écrire une boucle qui réémettrait chaque ligne à la main, et sans concaténer les fichiers dans une seule chaîne.

`simulate_logs(n)` est le même contrat sans disque. Le sujet demande un flux textuel simulé. Chaque tour produit une ligne JSON (`LOGIN_FAILED` la plupart du temps, `LOGIN_SUCCESS` tous les 23 événements) puis la rend par `yield`. Il n'existe pas de liste de `n` logs. Un `count` négatif lève `ValueError` avant de produire quoi que ce soit.

Le type renvoyé est vérifié ci-dessous : ce n'est pas une `list`. `sys.getsizeof` sur le générateur mesure l'objet générateur, pas le fichier. C'est précisément le point : l'objet qui représente « le reste du fichier » reste petit, que le fichier fasse 17 lignes ou plusieurs gigaoctets. Cette mesure ne prouve pas à elle seule l'absence de copie ; c'est le `yield` dans la source, affiché ensuite, qui le prouve.


In [5]:
import inspect
import sys

from secure_log_engine.ingestion import iter_file_lines, iter_sources, simulate_logs

print(inspect.getsource(iter_file_lines))
print(inspect.getsource(iter_sources))
print(inspect.getsource(simulate_logs))


def iter_file_lines(path: Path) -> Iterator[str]:
    """Lire un fichier ligne par ligne grâce à yield."""
    with Path(path).open("r", encoding="utf-8") as fichier:
        for ligne in fichier:
            yield ligne.rstrip("\r\n")

def iter_sources(paths: Iterable[Path]) -> Iterator[str]:
    """Lire plusieurs fichiers successivement grâce à yield from."""
    for path in paths:
        yield from iter_file_lines(path)

def simulate_logs(count: int) -> Iterator[str]:
    """Créer des logs de test au fur et à mesure, sans grande liste."""
    if count < 0:
        raise ValueError("Le nombre de logs ne peut pas être négatif.")

    debut = datetime(2026, 11, 1, 8, tzinfo=timezone.utc)

    for numero in range(count):
        ip_numero = numero % 200 + 1
        ip = f"192.0.2.{ip_numero}"
        evenement = "LOGIN_SUCCESS" if numero % 23 == 0 else "LOGIN_FAILED"
        niveau = "INFO" if evenement == "LOGIN_SUCCESS" else "WARNING"

        log = {
            "timestamp": (debut 

In [6]:
import sys

exemple = RACINE / "data" / "sample_1.log"
generateur = iter_file_lines(exemple)
print("Type :", type(generateur).__name__)
print("Taille de l'objet générateur (octets) :", sys.getsizeof(generateur))
print("Première ligne, tirée seule :")
print(next(generateur)[:80])
print("Le générateur n'est pas épuisé, une deuxième ligne est disponible :")
print(next(generateur)[:80])

plusieurs = iter_sources([
    RACINE / "data" / "sample_1.log",
    RACINE / "data" / "sample_2.log",
])
print("iter_sources est aussi un générateur :", type(plusieurs).__name__)

simulation = simulate_logs(3)
print("simulate_logs est un générateur :", type(simulation).__name__)
for ligne in simulation:
    print(ligne)

try:
    list(simulate_logs(-1))
except ValueError as erreur:
    print("Refus d'une simulation négative :", erreur)


Type : generator
Taille de l'objet générateur (octets) : 232
Première ligne, tirée seule :
{"timestamp":"2026-11-01T08:00:00Z","level":"info","ip":"192.0.2.1","user":"alic
Le générateur n'est pas épuisé, une deuxième ligne est disponible :
{"timestamp":"2026-11-01T08:01:00Z","level":"warning","ip":"192.0.2.1","user":"a
iter_sources est aussi un générateur : generator
simulate_logs est un générateur : generator
{"timestamp": "2026-11-01T08:00:00+00:00", "level": "INFO", "ip": "192.0.2.1", "user": "user_1", "event": "LOGIN_SUCCESS", "payload": {"code": "AUTH", "source": "simulation"}, "connection": ["192.0.2.1", 443, "TCP"]}
{"timestamp": "2026-11-01T08:00:01+00:00", "level": "WARNING", "ip": "192.0.2.2", "user": "user_2", "event": "LOGIN_FAILED", "payload": {"code": "AUTH", "source": "simulation"}, "connection": ["192.0.2.2", 443, "TCP"]}
{"timestamp": "2026-11-01T08:00:02+00:00", "level": "WARNING", "ip": "192.0.2.3", "user": "user_3", "event": "LOGIN_FAILED", "payload": {"code": "AUTH

## 8. Module 1 — parsing, normalisation, validation

`read_line_fields` reconnaît la forme de la ligne avant d'en extraire les champs.

1. Si le texte commence par `{` ou `[`, il est lu avec `json.loads`.
2. S'il commence par `(`, il est lu avec `ast.literal_eval`. Ce n'est pas `eval` : seuls les littéraux Python sont acceptés, pas un appel de fonction.
3. Sinon le texte est conservé tel quel.

Le `match` qui suit remplace une cascade `if` / `elif` sur les types.

- Un dictionnaire est déjà le bon format : il est renvoyé.
- Une séquence de 6 éléments est une ligne positionnelle `(date, niveau, ip, utilisateur, événement, payload)`.
- Une chaîne est découpée par `split("|", 5)`. Le `5` est important : le payload a le droit de contenir des barres, seul le début de la ligne est découpé en cinq séparateurs, donc six champs. Le dernier champ est reparsé comme JSON s'il l'est, sinon il reste une chaîne brute.
- Tout le reste lève `LogParseError`.

`parse_log_line` ajoute les contrôles de schéma : longueur maximale d'un million de caractères (une ligne absurde ne doit pas monopoliser le parseur), champs obligatoires, types texte, date ISO convertie en UTC, connexion vérifiée par `ipaddress`. Une liste de payload devient un tuple, pour que la valeur stockée soit immuable.

`normalize_record` ne « répare » pas une IP fausse. Elle enlève les espaces, met le niveau, l'événement et le protocole en majuscules, et trie les clés d'un payload dictionnaire en chaînes nettoyées. `error` et `ERROR` deviennent donc la même catégorie, ce que le fichier d'exemple provoque exprès.

`valid_record` est le filtre métier, distinct du filtre syntaxique. Une ligne peut être un JSON parfait et rester invalide : niveau hors de `{INFO, WARNING, ERROR, CRITICAL}`, utilisateur vide, événement vide, ou IP rejetée par `ipaddress.ip_address`. `pas-une-ip` emprunte ce chemin. `LIGNE_MAL_FORMEE` est rejetée plus tôt, par `LogParseError`.

`route_payload` classe le contenu, encore par `match`. L'ordre des cas est le comportement :

1. dictionnaire qui possède `level == "ERROR"` et une clé `code` → `JSON_ERREUR_<code>` ;
2. autre dictionnaire → `DICTIONNAIRE_JSON` ;
3. tuple `(str, int, str)` → `TUPLE_CONNEXION_<PROTOCOLE>` ;
4. autre tuple → `TUPLE_AUTRE` ;
5. chaîne → `CHAINE_BRUTE` ;
6. sinon → `AUTRE`.

Le cas 1 doit précéder le cas 2, sinon un JSON d'erreur serait absorbé par le cas général des dictionnaires.


In [7]:
import inspect
from secure_log_engine.parsing import read_line_fields, route_payload

print(inspect.getsource(read_line_fields))
print(inspect.getsource(route_payload))


def read_line_fields(ligne: str) -> dict[str, Any]:
    """Lire une ligne et renvoyer un dictionnaire de ses champs."""
    ligne = ligne.strip()

    try:
        if ligne.startswith(("{", "[")):
            contenu = json.loads(ligne)
        elif ligne.startswith("("):
            contenu = ast.literal_eval(ligne)  # Plus sûr que eval().
        else:
            contenu = ligne
    except (ValueError, SyntaxError, RecursionError) as erreur:
        raise LogParseError("Format de log incorrect") from erreur

    # match-case permet de reconnaître le type de données reçu.
    match contenu:
        case dict():
            return contenu
        case (list() | tuple()) if len(contenu) == 6:
            date, niveau, ip, utilisateur, evenement, payload = contenu
        case str():
            champs = contenu.split("|", 5)
            if len(champs) != 6:
                raise LogParseError("La ligne doit contenir 6 champs séparés par |")
            date, niveau, ip, utilisateur, ev

In [8]:
import json

from secure_log_engine.parsing import (
    LogParseError,
    normalize_record,
    parse_log_line,
    route_payload,
    valid_record,
)

exemples = {
    "JSON": json.dumps({
        "timestamp": "2026-11-01T08:00:00Z",
        "level": "warning",
        "ip": "192.0.2.1",
        "user": "alice",
        "event": "LOGIN_FAILED",
        "payload": {"reason": "bad_password"},
        "connection": ["192.0.2.1", 443, "tcp"],
    }),
    "barres": '2026-11-01T08:04:00Z|ERROR|203.0.113.66|bob|API_ERROR|{"level":"ERROR","code":500}',
    "tuple": "('2026-11-01T08:06:00Z', 'warning', '198.51.100.4', 'carol', 'CONNECTION', ('198.51.100.4', 22, 'SSH'))",
    "malformée": "LIGNE_MAL_FORMEE",
    "IP illégale": json.dumps({
        "timestamp": "2026-11-01T08:05:00Z",
        "level": "ERROR",
        "ip": "pas-une-ip",
        "user": "test",
        "event": "BAD_IP",
        "payload": "Rejet",
    }),
}

for nom, ligne in exemples.items():
    print(f"--- {nom} ---")
    try:
        log = normalize_record(parse_log_line(ligne))
    except LogParseError as erreur:
        print(f"LogParseError : {erreur}")
        continue
    print(
        f"valide={valid_record(log)} | {log.level} | {log.ip} | {log.user} | "
        f"{log.event} | payload={route_payload(log.payload)} | connexion={log.connection}"
    )

print("--- Classement de payloads isolés ---")
payloads = [
    {"level": "ERROR", "code": 403},
    {"action": "login"},
    ("198.51.100.4", 22, "ssh"),
    ("juste", "un", "triplet", "de", "texte"),
    "GET /health HTTP/1.1",
]
for payload in payloads:
    print(f"{route_payload(payload):28} <= {payload!r}")


--- JSON ---
valide=True | WARNING | 192.0.2.1 | alice | LOGIN_FAILED | payload=DICTIONNAIRE_JSON | connexion=('192.0.2.1', 443, 'TCP')
--- barres ---
valide=True | ERROR | 203.0.113.66 | bob | API_ERROR | payload=JSON_ERREUR_500 | connexion=None
--- tuple ---
valide=True | WARNING | 198.51.100.4 | carol | CONNECTION | payload=TUPLE_CONNEXION_SSH | connexion=None
--- malformée ---
LogParseError : La ligne doit contenir 6 champs séparés par |
--- IP illégale ---
valide=False | ERROR | pas-une-ip | test | BAD_IP | payload=CHAINE_BRUTE | connexion=None
--- Classement de payloads isolés ---
JSON_ERREUR_403              <= {'level': 'ERROR', 'code': 403}
DICTIONNAIRE_JSON            <= {'action': 'login'}
TUPLE_CONNEXION_SSH          <= ('198.51.100.4', 22, 'ssh')
TUPLE_AUTRE                  <= ('juste', 'un', 'triplet', 'de', 'texte')
CHAINE_BRUTE                 <= 'GET /health HTTP/1.1'


### Bilan sur les deux fichiers

La cellule suivante rejoue exactement la politique de `main.py`, sans passer par l'affichage CLI : ligne vide, erreur de parsing, enregistrement invalide, enregistrement gardé. Les totaux attendus sur ce jeu sont **17 lignes lues, 14 gardées, 2 rejetées, 1 vide**. Le test `test_donnees_du_projet` fige ce triplet `(14, 2, 1)`.


In [9]:
from collections import Counter

from secure_log_engine.ingestion import iter_sources
from secure_log_engine.parsing import (
    LogParseError,
    iter_valid_logs,
    normalize_record,
    parse_log_line,
    route_payload,
    valid_record,
)

fichiers = [RACINE / "data" / "sample_1.log", RACINE / "data" / "sample_2.log"]
total = vides = erreurs_syntaxe = invalides = 0
niveaux = Counter()
routes = Counter()
logs_valides = []

for ligne in iter_sources(fichiers):
    total += 1
    if not ligne.strip():
        vides += 1
        continue
    try:
        log = normalize_record(parse_log_line(ligne))
    except LogParseError:
        erreurs_syntaxe += 1
        continue
    if not valid_record(log):
        invalides += 1
        continue
    logs_valides.append(log)
    niveaux[log.level] += 1
    routes[route_payload(log.payload)] += 1

print(f"Lignes lues            : {total}")
print(f"Lignes vides           : {vides}")
print(f"Erreurs de syntaxe     : {erreurs_syntaxe}")
print(f"Schéma ou IP invalide  : {invalides}")
print(f"Logs gardés            : {len(logs_valides)}")
print("Niveaux :", dict(niveaux))
print("Formes de payload :", dict(routes))
print()
print("Journal gardé :")
for log in logs_valides:
    print(f"  {log.timestamp.time()}  {log.level:8}  {log.ip:16}  {log.user:6}  {log.event}")


Lignes lues            : 17
Lignes vides           : 1
Erreurs de syntaxe     : 1
Schéma ou IP invalide  : 1
Logs gardés            : 14
Niveaux : {'INFO': 3, 'WARNING': 8, 'ERROR': 2, 'CRITICAL': 1}
Formes de payload : {'DICTIONNAIRE_JSON': 3, 'CHAINE_BRUTE': 8, 'JSON_ERREUR_500': 1, 'TUPLE_CONNEXION_SSH': 1, 'JSON_ERREUR_E42': 1}

Journal gardé :
  08:00:00  INFO      192.0.2.1         alice   LOGIN_SUCCESS
  08:01:00  WARNING   192.0.2.1         alice   LOGIN_FAILED
  08:02:00  WARNING   192.0.2.1         alice   LOGIN_FAILED
  08:03:00  WARNING   192.0.2.1         alice   LOGIN_FAILED
  08:04:00  ERROR     203.0.113.66      bob     API_ERROR
  08:04:30  CRITICAL  203.0.113.66      bob     FIREWALL_BLOCK
  08:05:10  INFO      192.0.2.1         alice   LOGIN_SUCCESS
  08:06:00  WARNING   198.51.100.4      carol   CONNECTION
  08:06:30  INFO      198.51.100.5      david   HTTP_REQUEST
  08:07:00  WARNING   203.0.113.66      bob     LOGIN_FAILED
  08:07:30  WARNING   203.0.113.66      

## 9. Module 2 — pipeline fonctionnel

Le sujet demande une chaîne de transformations sur l'itérateur, pas une nouvelle boucle impérative.

`filtre_lignes_vides` retourne `filter(lambda ligne: ligne.strip() != "", flux)`. `filter` ne parcourt pas le flux au moment de l'appel. Il rend un itérateur qui posera la question « cette ligne est-elle vide ? » à chaque demande. La lambda est le prédicat : une fonction anonyme, parce que le critère tient en une expression et n'a pas besoin d'un nom dans le reste du module.

`normaliser_flux` retourne `map(normaliser_un_log, flux)`. Même paresse. `normaliser_un_log` est une fonction nommée passée à `map` : c'est déjà le geste « fonction comme valeur », repris plus franchement par le moteur de règles.

`filtre_logs_valide` applique `valid_record` de la même façon, après la normalisation. L'ordre compte. `valid_record` compare le niveau à `NIVEAUX_VALIDES`, qui est en majuscules. Filtrer avant de normaliser rejetterait les lignes `info` du jeu d'exemple, qui sont pourtant légitimes.

`router_payload` dans `pipeline.py` est un second `match`, volontairement parallèle à celui de `parsing.py`. Il ne connaît pas le cas `TUPLE_AUTRE` : un tuple qui n'est pas `(str, int, str)` tombe dans le cas par défaut `AUTRE`. Le classement fin utilisé par `main.py` est celui de `parsing.route_payload`.

La cellule construit la chaîne et n'appelle `list` qu'à la fin, pour montrer les types intermédiaires. Les rejets syntaxiques sont isolés dans un petit générateur local : `pipeline.py` ne les attrape pas, `main.py` si. C'est documenté comme limite à la section 15.


In [10]:
import inspect

from secure_log_engine import pipeline

print(inspect.getsource(pipeline))


from .parsing import normalize_record,valid_record

def filtre_lignes_vides(flux_lignes):
    return filter(lambda ligne: ligne.strip() != "",flux_lignes)

def filtre_logs_valide(flux_logs):
    return filter(lambda log: valid_record(log),flux_logs)

def normaliser_un_log(log):
    log = normalize_record(log)
    return log

def normaliser_flux(flux_logs):
    return map(normaliser_un_log,flux_logs)

def formater_log(log):
    return f"[{log.level}] {log.timestamp} | {log.ip} | {log.user} | {log.event}"

def router_payload(payload):
    match payload:
        case {"level": "ERROR", "code": code}:
            return f"JSON_ERREUR_{code}"
        case dict():
            return "DICTIONNAIRE_JSON"
        case (str(), int(), str() as protocole):
            return f"TUPLE_CONNEXION_{protocole.upper()}"
        case str():
            return "CHAINE_BRUTE"
        case _:
            return "AUTRE"



In [11]:
from secure_log_engine.ingestion import iter_sources
from secure_log_engine.parsing import LogParseError, parse_log_line
from secure_log_engine.pipeline import (
    filtre_lignes_vides,
    filtre_logs_valide,
    formater_log,
    normaliser_flux,
    router_payload,
)

def sans_erreur_de_syntaxe(lignes):
    for ligne in lignes:
        try:
            yield parse_log_line(ligne)
        except LogParseError as erreur:
            print(f"écarté par le parseur : {ligne[:48]!r} ({erreur})")

fichiers = [RACINE / "data" / "sample_1.log", RACINE / "data" / "sample_2.log"]
brutes = iter_sources(fichiers)
non_vides = filtre_lignes_vides(brutes)
parsees = sans_erreur_de_syntaxe(non_vides)
normalisees = normaliser_flux(parsees)
valides = filtre_logs_valide(normalisees)
formatees = map(formater_log, valides)

print("filter :", type(non_vides).__name__)
print("map    :", type(normalisees).__name__)
print("Rien n'a encore été lu : les objets ci-dessus sont des itérateurs paresseux.")
print()
for ligne in formatees:
    print(ligne)

print()
print("Routage du module 2 :")
for payload in (
    {"level": "ERROR", "code": "E42"},
    {"detail": "accès"},
    ("198.51.100.4", 22, "SSH"),
    "Mot de passe incorrect",
):
    print(f"  {router_payload(payload):28} <= {payload!r}")


filter : filter
map    : map
Rien n'a encore été lu : les objets ci-dessus sont des itérateurs paresseux.

[INFO] 2026-11-01 08:00:00+00:00 | 192.0.2.1 | alice | LOGIN_SUCCESS
[WARNING] 2026-11-01 08:01:00+00:00 | 192.0.2.1 | alice | LOGIN_FAILED
[WARNING] 2026-11-01 08:02:00+00:00 | 192.0.2.1 | alice | LOGIN_FAILED
[WARNING] 2026-11-01 08:03:00+00:00 | 192.0.2.1 | alice | LOGIN_FAILED
[ERROR] 2026-11-01 08:04:00+00:00 | 203.0.113.66 | bob | API_ERROR
[CRITICAL] 2026-11-01 08:04:30+00:00 | 203.0.113.66 | bob | FIREWALL_BLOCK
écarté par le parseur : 'LIGNE_MAL_FORMEE' (La ligne doit contenir 6 champs séparés par |)
[INFO] 2026-11-01 08:05:10+00:00 | 192.0.2.1 | alice | LOGIN_SUCCESS
[WARNING] 2026-11-01 08:06:00+00:00 | 198.51.100.4 | carol | CONNECTION
[INFO] 2026-11-01 08:06:30+00:00 | 198.51.100.5 | david | HTTP_REQUEST
[WARNING] 2026-11-01 08:07:00+00:00 | 203.0.113.66 | bob | LOGIN_FAILED
[WARNING] 2026-11-01 08:07:30+00:00 | 203.0.113.66 | bob | LOGIN_FAILED
[WARNING] 2026-11-01 0

## 10. Module 3 — ensembles, compteurs, force brute

### Ensembles

Une adresse qui a produit un `ERROR` ou un `CRITICAL` entre dans un `set`. L'appartenance et l'unicité sont le contrat d'un ensemble : `203.0.113.66` a plusieurs lignes en erreur, elle n'est stockée qu'une fois. `ips_en_erreur` l'écrit en compréhension. `main.py` le recalcule pendant le parcours, avec le même critère, pour ne pas consommer deux fois le générateur.

La blacklist est un `frozenset`. On peut calculer `|`, `&`, `-` et `^` avec un `set`, mais on ne peut pas appeler `.add` sur le `frozenset`. La liste de référence ne peut pas être élargie par accident au milieu d'un calcul.

Sur le jeu fourni, les quatre opérations se lisent ainsi. Les IP actives sont celles des logs valides. La blacklist est `{203.0.113.66, 198.51.100.99}`.

| Opération | Lecture |
| --- | --- |
| `actives \| blacklist` | toutes les IP dont on parle, vues ou surveillées |
| `actives & blacklist` | IP à la fois vues et interdites : ici seulement `203.0.113.66` |
| `actives - blacklist` | IP vues qui ne sont pas interdites |
| `actives ^ blacklist` | IP qui sont d'un seul des deux côtés. `198.51.100.99` y figure parce qu'elle est surveillée sans avoir parlé. Les IP actives non blacklistées y figurent aussi |

`198.51.100.99` absente des logs est le cas intéressant. Une intersection ne la montre pas. Une union et une différence symétrique si. Confondre « IP surveillée » et « IP vue » serait une erreur d'analyse.

### Compteurs

`defaultdict(int)` traite une clé absente comme `0`. `par_niveau[log.level] += 1` ne lève pas `KeyError` au premier `CRITICAL`. `compter_occurrences` tient le même compte par niveau et par utilisateur. `main.py` ne compte que les niveaux, dans son propre `defaultdict`, pendant l'unique parcours du flux.

### Force brute

`detecter_force_brute` cherche `seuil` échecs de connexion consécutifs pour le couple `(IP, utilisateur)`. Le seuil par défaut est 3. L'état par couple est un `deque(maxlen=seuil)` : à chaque login, l'événement entre, et le plus ancien sort tout seul quand la fenêtre est pleine. On ne conserve donc pas l'historique complet de l'utilisateur.

Le flux est lu par lots de `taille_cycle` (50) grâce à `islice`, dans un `while`. Un lot vide termine la fonction. Ce n'est pas une optimisation utile sur 14 lignes ; c'est la forme demandée d'une boucle `while` qui avance dans un itérateur sans le convertir d'abord en liste géante.

À l'intérieur du lot, le `for` applique trois instructions de contrôle, toutes exigées par le sujet.

- `continue` si l'événement n'est ni `LOGIN_SUCCESS` ni `LOGIN_FAILED`. Un `DISK_ERROR` ou un `FIREWALL_BLOCK` ne remet pas la fenêtre à zéro et n'en fait pas partie.
- `continue` après un `LOGIN_SUCCESS`, une fois le drapeau `deja_alertee` remis à faux. Un succès casse la série et autorise une future alerte.
- `continue` si la fenêtre n'est pas pleine, ou si ce couple a déjà été signalé pour la série en cours.
- `break` dans la boucle interne, dès qu'un événement de la fenêtre n'est pas un échec.
- `else` de cette boucle interne : il s'exécute seulement si le `break` n'a pas eu lieu, donc seulement si les `seuil` derniers logins sont tous des échecs. C'est là que l'alerte est créée. Le drapeau passe alors à vrai, et le prochain échec de la même série est ignoré.

Ce `else` n'est pas le `else` d'un `if`. Il appartient au `for`. C'est le point du sujet : la boucle a un succès complet, distinct d'une sortie anticipée.

Chronologie d'alice (`192.0.2.1`), fenêtre de 3 :

| Heure | Événement | Fenêtre après l'ajout | Décision |
| --- | --- | --- | --- |
| 08:00 | succès | `[succès]` | drapeau réarmé, pas d'alerte |
| 08:01 | échec | `[succès, échec]` | trop court |
| 08:02 | échec | `[succès, échec, échec]` | plein, mais le `break` voit le succès, pas de `else` |
| 08:03 | échec | `[échec, échec, échec]` | pas de `break`, le `else` crée l'alerte |
| 08:05 | succès | `[échec, échec, succès]` | drapeau réarmé |

Bob (`203.0.113.66`) n'a aucun login avant 08:07. Ses trois premiers `LOGIN_FAILED` remplissent une fenêtre homogène à 08:08 et créent la seconde alerte. Le quatrième échec, à 08:08:30, tombe sur `deja_alertee` et fait `continue`. Les événements `API_ERROR` et `FIREWALL_BLOCK` du même bob, plus tôt, ont été sautés par le premier `continue` : ils ne vident pas une fenêtre de login, parce qu'ils n'y entrent pas.


In [12]:
import inspect
from secure_log_engine.analysis import (
    comparer_blacklist,
    compter_occurrences,
    detecter_force_brute,
    ips_en_erreur,
)

print(inspect.getsource(comparer_blacklist))
print(inspect.getsource(detecter_force_brute))


def comparer_blacklist(
    ips_actives: set[str],
    blacklist: frozenset[str],
) -> dict[str, set[str]]:
    blacklist_immuable = frozenset(blacklist)
    return {
        "union": ips_actives | blacklist_immuable,
        "intersection": ips_actives & blacklist_immuable,
        "difference": ips_actives - blacklist_immuable,
        "difference_symetrique": ips_actives ^ blacklist_immuable,
    }

def detecter_force_brute(
    logs: Iterable[LogRecord],
    seuil: int = 3,
    taille_cycle: int = 20,
) -> list[Finding]:
    if seuil < 1 or taille_cycle < 1:
        raise ValueError("seuil et taille_cycle doivent être >= 1.")

    historique: defaultdict[tuple[str, str], deque[str]] = defaultdict(
        lambda: deque(maxlen=seuil)
    )
    deja_alertee: defaultdict[tuple[str, str], bool] = defaultdict(bool)
    alertes: list[Finding] = []

    flux = iter(logs)
    lot = list(islice(flux, taille_cycle))

    while lot:
        for log in lot:
            if log.event not in {"LO

In [13]:
BLACKLIST = frozenset({"203.0.113.66", "198.51.100.99"})

try:
    BLACKLIST.add("192.0.2.1")
except AttributeError as erreur:
    print("Le frozenset refuse la mutation :", type(erreur).__name__)

ips_actives = {log.ip for log in logs_valides}
print("IP actives :", sorted(ips_actives))
print("IP en erreur ou critiques :", sorted(ips_en_erreur(logs_valides)))

ensembles = comparer_blacklist(ips_actives, BLACKLIST)
libelles = {
    "union": "actives | blacklist",
    "intersection": "actives & blacklist",
    "difference": "actives - blacklist",
    "difference_symetrique": "actives ^ blacklist",
}
for cle, libelle in libelles.items():
    print(f"{libelle:28} {sorted(ensembles[cle])}")

par_niveau, par_utilisateur = compter_occurrences(logs_valides)
print("Occurrences par niveau     :", dict(par_niveau))
print("Occurrences par utilisateur:", dict(par_utilisateur))

print()
print("Histogramme des niveaux")
maximum = max(par_niveau.values())
for niveau in ("INFO", "WARNING", "ERROR", "CRITICAL"):
    barre = "█" * par_niveau[niveau]
    print(f"  {niveau:8} {par_niveau[niveau]:2} {barre}")
print(f"  (échelle : un carré = un log, maximum {maximum})")


Le frozenset refuse la mutation : AttributeError
IP actives : ['192.0.2.1', '198.51.100.4', '198.51.100.5', '203.0.113.66']
IP en erreur ou critiques : ['198.51.100.4', '203.0.113.66']
actives | blacklist          ['192.0.2.1', '198.51.100.4', '198.51.100.5', '198.51.100.99', '203.0.113.66']
actives & blacklist          ['203.0.113.66']
actives - blacklist          ['192.0.2.1', '198.51.100.4', '198.51.100.5']
actives ^ blacklist          ['192.0.2.1', '198.51.100.4', '198.51.100.5', '198.51.100.99']
Occurrences par niveau     : {'INFO': 3, 'WARNING': 8, 'ERROR': 2, 'CRITICAL': 1}
Occurrences par utilisateur: {'alice': 5, 'bob': 6, 'carol': 2, 'david': 1}

Histogramme des niveaux
  INFO      3 ███
  WARNING   8 ████████
  ERROR     2 ██
  CRITICAL  1 █
  (échelle : un carré = un log, maximum 8)


In [14]:
from collections import defaultdict, deque

print("Rejeu de la fenêtre de login, seuil = 3")
print(f"{'heure':10} {'utilisateur':8} {'événement':16} fenêtre après coup")
fenetres = defaultdict(lambda: deque(maxlen=3))
for log in logs_valides:
    if log.event not in {"LOGIN_SUCCESS", "LOGIN_FAILED"}:
        print(f"{log.timestamp.strftime('%H:%M:%S'):10} {log.user:8} {log.event:16} (ignoré, hors login)")
        continue
    cle = (log.ip, log.user)
    fenetres[cle].append(log.event)
    contenu = " ".join(evenement.removeprefix("LOGIN_") for evenement in fenetres[cle])
    print(f"{log.timestamp.strftime('%H:%M:%S'):10} {log.user:8} {log.event:16} [{contenu}]")

print()
print("Alertes renvoyées par detecter_force_brute :")
for alerte in detecter_force_brute(logs_valides, seuil=3):
    print(f"  {alerte.timestamp}  {alerte.ip}  {alerte.user}  règle={alerte.rule!r}")


Rejeu de la fenêtre de login, seuil = 3
heure      utilisateur événement        fenêtre après coup
08:00:00   alice    LOGIN_SUCCESS    [SUCCESS]
08:01:00   alice    LOGIN_FAILED     [SUCCESS FAILED]
08:02:00   alice    LOGIN_FAILED     [SUCCESS FAILED FAILED]
08:03:00   alice    LOGIN_FAILED     [FAILED FAILED FAILED]
08:04:00   bob      API_ERROR        (ignoré, hors login)
08:04:30   bob      FIREWALL_BLOCK   (ignoré, hors login)
08:05:10   alice    LOGIN_SUCCESS    [FAILED FAILED SUCCESS]
08:06:00   carol    CONNECTION       (ignoré, hors login)
08:06:30   david    HTTP_REQUEST     (ignoré, hors login)
08:07:00   bob      LOGIN_FAILED     [FAILED]
08:07:30   bob      LOGIN_FAILED     [FAILED FAILED]
08:08:00   bob      LOGIN_FAILED     [FAILED FAILED FAILED]
08:08:30   bob      LOGIN_FAILED     [FAILED FAILED FAILED]
08:09:00   carol    DISK_ERROR       (ignoré, hors login)

Alertes renvoyées par detecter_force_brute :
  2026-11-01 08:03:00+00:00  192.0.2.1  alice  règle=''
  2026-

## 11. Module 4 — moteur de règles

`apply_security_rules(*rules, **config)` est la fonction maître demandée par le sujet.

Les règles arrivent en arguments positionnels variables. L'étoile ne « groupe pas des logs » : elle groupe des fonctions. Chaque élément est testé avec `callable`. Une chaîne, un nombre ou `None` lève `SecurityRuleError`, qui hérite de `TypeError`. Zéro règle lève la même exception : un moteur appelé pour ne rien faire est une erreur d'appel, pas un succès vide.

La configuration arrive en arguments nommés. `logs` est obligatoire. Les autres clés (`seuil`, `blacklist`, `evenements_sensibles`, ou n'importe quel nom inventé par une règle future) sont transmises telles quelles à chaque règle, sous la forme `regle(enregistrements, **config)`. La règle lit ce dont elle a besoin avec `config.get` et ignore le reste. C'est ce qui permet d'appliquer trois règles avec un seul dictionnaire de paramètres, sans adapter la signature du moteur à chaque nouvelle règle.

Les fonctions sont des objets de première classe au sens précis du cours : elles sont stockées dans le tuple `rules`, puis appelées. Rien dans le moteur ne cite `regle_force_brute` par son nom. Le lien est fait par l'appelant, dans `main.py` :

```python
apply_security_rules(
    regle_force_brute,
    regle_ip_blacklist,
    regle_evenements_sensibles,
    logs=logs_valides,
    seuil=3,
    blacklist=BLACKLIST,
    evenements_sensibles=evenements_sensibles,
)
```

`regle_force_brute` ne réécrit pas l'algorithme. Elle appelle `detecter_force_brute` et traduit chaque `Finding` en constat étiqueté `force_brute`, avec le seuil dans le détail. `regle_ip_blacklist` reconstruit un `frozenset` à partir de la config et émet un constat par ligne dont l'IP y figure. `regle_evenements_sensibles` compare `log.event` à l'ensemble reçu. Sans cet argument, elle ne surveille que `FIREWALL_BLOCK`.

Le moteur transforme l'itérable de logs en `tuple` avant la première règle. Un générateur ne se parcourt qu'une fois : la deuxième règle verrait un flux déjà vide. Le tuple est une copie des enregistrements déjà parsés, pas une relecture du fichier. Sur un très grand volume, le coût mémoire se déplace donc ici, après filtrage. La section 15 le dit explicitement.

Une règle doit renvoyer un itérable de `Finding`. Si ce contrat casse, `list.extend` lève `TypeError`, et le moteur le transforme en `SecurityRuleError` en nommant la règle. L'erreur d'origine est chaînée avec `from erreur`, elle n'est pas perdue.

Les trois règles sur le jeu fourni produisent **10 constats** : 2 forces brutes (alice, bob), 6 lignes de l'IP `203.0.113.66`, 2 événements sensibles (`FIREWALL_BLOCK`, `DISK_ERROR`). `198.51.100.99` n'en produit aucun : elle est sur la blacklist, mais aucune ligne ne la porte.


In [15]:
import inspect
from secure_log_engine.rules import apply_security_rules

print(inspect.getsource(apply_security_rules))


def apply_security_rules(*rules: SecurityRule, **config: Any) -> list[Finding]:
    """Applique un nombre variable de règles, configurées via **kwargs.

    Les règles sont passées en *args et appelées comme des fonctions ordinaires.
    La configuration (seuil, blacklist, événements sensibles, etc.) arrive dans
    **kwargs. Le flux de logs est obligatoire : ``logs=...``.
    """
    if not rules:
        raise SecurityRuleError("Aucune règle de sécurité fournie.")
    if "logs" not in config:
        raise SecurityRuleError("Le paramètre 'logs' est obligatoire.")

    logs = config.pop("logs")
    try:
        enregistrements = tuple(logs)
    except TypeError as erreur:
        raise SecurityRuleError("'logs' doit être un itérable de LogRecord.") from erreur

    resultats: list[Finding] = []
    for regle in rules:
        if not callable(regle):
            raise SecurityRuleError(f"Règle non appelable : {regle!r}")

        produit = regle(enregistrements, **config)
        try:


In [16]:
from collections import Counter

from secure_log_engine.models import Finding
from secure_log_engine.rules import (
    SecurityRuleError,
    apply_security_rules,
    regle_evenements_sensibles,
    regle_force_brute,
    regle_ip_blacklist,
)

def regle_critique(logs, **config):
    """Règle locale : prouve qu'une fonction quelconque peut entrer dans *args."""
    for log in logs:
        if log.level == "CRITICAL":
            yield Finding(
                log.timestamp, log.ip, log.user,
                rule="niveau_critique",
                detail=log.event,
            )

constats = apply_security_rules(
    regle_force_brute,
    regle_ip_blacklist,
    regle_evenements_sensibles,
    regle_critique,
    logs=logs_valides,
    seuil=3,
    blacklist=BLACKLIST,
    evenements_sensibles=frozenset({"FIREWALL_BLOCK", "DISK_ERROR"}),
)

print("Nombre de constats par règle :", dict(Counter(item.rule for item in constats)))
for constat in constats:
    print(
        f"  [{constat.rule:20}] {constat.timestamp.strftime('%H:%M:%S')}  "
        f"{constat.ip:16} {constat.user:6} {constat.detail}"
    )

print()
print("Appels refusés :")
for titre, appel in (
    ("aucune règle", lambda: apply_security_rules(logs=[])),
    ("logs manquant", lambda: apply_security_rules(regle_ip_blacklist)),
    ("règle non appelable", lambda: apply_security_rules("blacklist", logs=[])),
    ("logs non itérable", lambda: apply_security_rules(regle_ip_blacklist, logs=123)),
):
    try:
        appel()
    except SecurityRuleError as erreur:
        print(f"  {titre:22} -> {type(erreur).__name__}: {erreur}")


Nombre de constats par règle : {'force_brute': 2, 'blacklist': 6, 'evenement_sensible': 2, 'niveau_critique': 1}
  [force_brute         ] 08:03:00  192.0.2.1        alice  3 échecs de connexion consécutifs
  [force_brute         ] 08:08:00  203.0.113.66     bob    3 échecs de connexion consécutifs
  [blacklist           ] 08:04:00  203.0.113.66     bob    IP présente dans la blacklist
  [blacklist           ] 08:04:30  203.0.113.66     bob    IP présente dans la blacklist
  [blacklist           ] 08:07:00  203.0.113.66     bob    IP présente dans la blacklist
  [blacklist           ] 08:07:30  203.0.113.66     bob    IP présente dans la blacklist
  [blacklist           ] 08:08:00  203.0.113.66     bob    IP présente dans la blacklist
  [blacklist           ] 08:08:30  203.0.113.66     bob    IP présente dans la blacklist
  [evenement_sensible  ] 08:04:30  203.0.113.66     bob    FIREWALL_BLOCK
  [evenement_sensible  ] 08:09:00  198.51.100.4     carol  DISK_ERROR
  [niveau_critique     

## 12. Module 4 — audit par introspection

`auditer_contexte` répond à la demande d'un module qui inspecte l'état d'exécution avec `globals()` et `locals()`.

`globals()`, appelé dans `audit.py`, est le dictionnaire du module d'audit : les fonctions `auditer_contexte` et `formater_rapport_audit`, l'aide `_decrire`, et les noms importés (`datetime`, `timezone`, `Any`, `Mapping`). Ce n'est pas l'ensemble de l'interpréteur. Chaque module a sa table globale.

`locals()`, appelé à la fin de la fonction, est la table des variables de cet appel : l'horodatage, la copie de la portée demandée, le nombre de variables de l'appelant, et le dictionnaire global qui vient d'être capturé. L'appel est placé après ces affectations. Un `locals()` plus haut n'aurait pas encore vu `variables_globales`.

Le troisième volet est optionnel. `main.py` appelle `auditer_contexte(locals())` depuis `traiter_les_logs`. La fonction d'audit ne peut pas lire toute seule les variables de son appelant : on les lui passe. `_decrire` ne retient que le nom et le type, trie les noms, et écarte les noms magiques `__...__`. Le rapport reste lisible et n'affiche pas la valeur d'une liste de logs.

La cellule appelle l'audit depuis une fonction étroite, pour que la portée de l'appelant soit celle de la démonstration et non l'ensemble du notebook. Le pipeline complet, plus bas, montre ensuite le même audit tel que `main.py` le produit, avec les compteurs, les ensembles et les constats de la vraie exécution.


In [17]:
import inspect
from secure_log_engine.audit import auditer_contexte, formater_rapport_audit

print(inspect.getsource(auditer_contexte))
print(inspect.getsource(formater_rapport_audit))


def auditer_contexte(portee_externe: Mapping[str, Any] | None = None) -> dict[str, Any]:
    """Exporte les variables chargées en mémoire pour l'audit et le débogage.

    ``globals()`` décrit le module d'audit. ``locals()`` décrit cette fonction
    au moment de l'export. ``portee_externe`` permet d'y ajouter la portée de
    l'appelant (en général ``locals()`` depuis ``main``).
    """
    horodatage = datetime.now(timezone.utc).isoformat()
    portee_demandee = {} if portee_externe is None else dict(portee_externe)
    nombre_variables_appelant = len(portee_demandee)
    variables_globales = globals()
    # Instantané pris après la création des variables locales ci-dessus.
    variables_locales = locals()

    return {
        "horodatage": horodatage,
        "nombre_variables_appelant": nombre_variables_appelant,
        "globales": _decrire(variables_globales),
        "locales": _decrire(variables_locales),
        "appelant": _decrire(portee_demandee),
    }

def formater_rappo

In [18]:
from secure_log_engine.audit import auditer_contexte, formater_rapport_audit

def demonstration_audit():
    seuil_demo = 3
    evenements_surveilles = frozenset({"FIREWALL_BLOCK", "DISK_ERROR"})
    nombre_logs = len(logs_valides)
    rapport = auditer_contexte(locals())
    print(formater_rapport_audit(rapport))
    return rapport

rapport_demo = demonstration_audit()
print()
print("Le module d'audit se voit lui-même dans globals() :",
      "auditer_contexte" in rapport_demo["globales"])
print("La fonction de démo se voit dans la portée appelante :",
      "seuil_demo" in rapport_demo["appelant"])
print("Le seuil y est un entier :", rapport_demo["appelant"]["seuil_demo"])


Horodatage : 2026-11-12T10:32:36.516180+00:00
Variables transmises par l'appelant : 3
Portée globale du module d'audit (7) :
  - Any: _AnyMeta
  - Mapping: _SpecialGenericAlias
  - _decrire: function
  - auditer_contexte: function
  - datetime: type
  - formater_rapport_audit: function
  - timezone: type
Portée locale de l'audit (5) :
  - horodatage: str
  - nombre_variables_appelant: int
  - portee_demandee: dict
  - portee_externe: dict
  - variables_globales: dict
Portée de l'appelant (3) :
  - evenements_surveilles: frozenset
  - nombre_logs: int
  - seuil_demo: int

Le module d'audit se voit lui-même dans globals() : True
La fonction de démo se voit dans la portée appelante : True
Le seuil y est un entier : int


## 13. Pipeline complet de `main.py`

`traiter_les_logs` est le seul endroit qui assemble les quatre modules pour un utilisateur.

1. Il reçoit un itérable de lignes. Dans `main()`, ce sont soit `iter_sources` sur les deux fichiers, soit `simulate_logs` si l'option `--simulate N` est présente. `argparse` ne charge rien : il lit seulement les arguments.
2. Un générateur interne, `flux_valide`, tire les lignes une à une. Il compte le total, saute les vides, attrape `LogParseError`, rejette `valid_record` faux, et met à jour le `set` d'IP, le `set` d'IP en erreur et le `defaultdict` des niveaux.
3. Les dix premiers logs valides sont imprimés avec le classement de payload. La limite est un choix d'affichage, pas une limite de traitement.
4. Chaque log valide est aussi ajouté à `logs_valides`, puis cédé par `yield`. `detecter_force_brute(flux_valide())` est le consommateur qui fait avancer ce générateur. Tant que cette fonction n'a pas parcouru le flux, les compteurs restent à zéro. C'est le comportement normal d'un générateur, et c'est pour cela que le bilan est imprimé après l'appel, jamais avant.
5. `comparer_blacklist` tourne ensuite : les ensembles d'IP sont déjà remplis, parce que le générateur a été épuisé.
6. `apply_security_rules` reçoit les trois règles et la configuration. Il reparcourt `logs_valides`.
7. `auditer_contexte(locals())` photographie la fonction `traiter_les_logs` à cet instant : alertes, ensembles, constats, compteurs.

La sortie ci-dessous est celle du programme, pas une réécriture. Les ensembles Python n'ont pas d'ordre d'affichage stable : l'identité des IP compte, pas leur position dans les accolades.


In [19]:
from main import BLACKLIST as BLACKLIST_MAIN
from main import traiter_les_logs
from secure_log_engine.ingestion import iter_sources

print("Blacklist utilisée par main.py :", sorted(BLACKLIST_MAIN))
print()
traiter_les_logs(iter_sources([
    RACINE / "data" / "sample_1.log",
    RACINE / "data" / "sample_2.log",
]))


Blacklist utilisée par main.py : ['198.51.100.99', '203.0.113.66']

--- Démarrage de la lecture des logs ---
[INFO] 2026-11-01 08:00:00+00:00 | IP: 192.0.2.1 | User: alice | Event: LOGIN_SUCCESS (DICTIONNAIRE_JSON)
[WARNING] 2026-11-01 08:01:00+00:00 | IP: 192.0.2.1 | User: alice | Event: LOGIN_FAILED (DICTIONNAIRE_JSON)
[WARNING] 2026-11-01 08:02:00+00:00 | IP: 192.0.2.1 | User: alice | Event: LOGIN_FAILED (CHAINE_BRUTE)
[WARNING] 2026-11-01 08:03:00+00:00 | IP: 192.0.2.1 | User: alice | Event: LOGIN_FAILED (CHAINE_BRUTE)
[ERROR] 2026-11-01 08:04:00+00:00 | IP: 203.0.113.66 | User: bob | Event: API_ERROR (JSON_ERREUR_500)
[CRITICAL] 2026-11-01 08:04:30+00:00 | IP: 203.0.113.66 | User: bob | Event: FIREWALL_BLOCK (DICTIONNAIRE_JSON)
[INFO] 2026-11-01 08:05:10+00:00 | IP: 192.0.2.1 | User: alice | Event: LOGIN_SUCCESS (CHAINE_BRUTE)
[WARNING] 2026-11-01 08:06:00+00:00 | IP: 198.51.100.4 | User: carol | Event: CONNECTION (TUPLE_CONNEXION_SSH)
[INFO] 2026-11-01 08:06:30+00:00 | IP: 198.51

### Lecture du bilan affiché

Sur ce jeu, la sortie se décompose en quatre blocs.

**Traitement.** 17 lignes entrent dans le générateur. 14 sont des `LogRecord` valides. 2 sont rejetées (la ligne sans format, et l'IP `pas-une-ip`). La ligne vide est comptée dans le total puis sautée sans être une erreur de parsing. Les niveaux se répartissent en 3 `INFO`, 8 `WARNING`, 2 `ERROR`, 1 `CRITICAL`.

**Sécurité, module 3.** Les IP en erreur ou critiques sont `203.0.113.66` (bob) et `198.51.100.4` (carol, `DISK_ERROR`). Deux alertes de force brute : alice à 08:03, bob à 08:08. L'intersection avec la blacklist ne contient que `203.0.113.66`. La différence symétrique contient `198.51.100.99`, l'adresse surveillée qui n'a aucun log.

**Règles, module 4.** Les 10 constats reprennent les deux forces brutes, les six lignes de l'IP interdite, puis `FIREWALL_BLOCK` et `DISK_ERROR`. La force brute apparaît deux fois dans la sortie complète, une fois comme alerte du module 3 et une fois comme constat de règle. Ce n'est pas un double comptage dans une même liste : le module 4 réutilise l'algorithme du module 3 comme une fonction branchée sur le moteur.

**Audit.** La portée appelante contient notamment `alertes`, `constats`, `ensembles`, `ips_actives`, `ips_erreur`, `par_niveau`, `logs_valides` et le générateur `flux_valide`. Ce sont les variables réellement présentes dans `traiter_les_logs` au moment de l'export.

`python main.py --simulate 50` suit le même chemin avec un flux fabriqué. La simulation n'est pas rejouée ici : elle ne sert pas à illustrer le jeu fourni, et elle produirait cinquante lignes du même schéma.


## 14. Tests

Les tests utilisent `unittest`, donc la bibliothèque standard. `tests/test_partie1.py` couvre l'ingestion (plusieurs fichiers, générateur, refus d'un nombre négatif) et le parsing (JSON normalisé, ligne à barres, tuple, lignes invalides, totaux du jeu). `tests/test_partie4.py` couvre le moteur : ordre des fonctions passées en `*args`, transmission des `**kwargs`, rejets, résultat des trois règles sur les fichiers d'exemple, et présence des portées `globals` / `locals` / appelant dans le rapport d'audit.

La cellule lance la même commande que le README.


In [20]:
import subprocess
import sys

resultat = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "tests", "-v"],
    cwd=RACINE,
    capture_output=True,
    text=True,
)
print(resultat.stdout)
if resultat.stderr:
    print(resultat.stderr)
print("Code de retour :", resultat.returncode)
if resultat.returncode != 0:
    raise SystemExit(resultat.returncode)



test_la_simulation_ne_construit_pas_de_liste (test_partie1.IngestionTests.test_la_simulation_ne_construit_pas_de_liste) ... ok
test_lire_plusieurs_fichiers (test_partie1.IngestionTests.test_lire_plusieurs_fichiers) ... ok
test_donnees_du_projet (test_partie1.ParsingTests.test_donnees_du_projet) ... ok
test_json_et_normalisation (test_partie1.ParsingTests.test_json_et_normalisation) ... ok
test_lignes_invalides (test_partie1.ParsingTests.test_lignes_invalides) ... ok
test_texte_avec_barres (test_partie1.ParsingTests.test_texte_avec_barres) ... ok
test_tuple_python (test_partie1.ParsingTests.test_tuple_python) ... ok
test_globals_et_locals_sont_exportes (test_partie4.AuditTests.test_globals_et_locals_sont_exportes) ... ok
test_regles_passees_en_args_et_config_en_kwargs (test_partie4.MoteurReglesTests.test_regles_passees_en_args_et_config_en_kwargs) ... ok
test_regles_sur_les_logs_du_projet (test_partie4.MoteurReglesTests.test_regles_sur_les_logs_du_projet) ... ok
test_rejet_si_regle_ou_

## 15. Limites connues

Ces limites ne cassent pas le cahier des charges. Elles décrivent ce que le code fait vraiment, pour qu'un lecteur ne les prenne pas pour des propriétés non écrites.

**`pipeline.py` n'est pas sur le chemin de `main.py`.** La chaîne `filter` / `map` existe et ce notebook l'exécute. L'exécutable de démonstration réécrit parse, normalisation et validation dans `flux_valide`, parce qu'il doit compter les rejets en même temps. Les deux chemins appliquent les mêmes fonctions de `parsing.py`. Ils peuvent diverger si l'un est modifié sans l'autre.

**Le fichier est en flux, les logs valides ne le restent pas jusqu'au bout.** `iter_file_lines` ne charge pas le texte. `traiter_les_logs` ajoute ensuite chaque `LogRecord` valide à une liste, et `apply_security_rules` en fait un tuple pour que chaque règle puisse reparcourir les données. Sur le jeu d'exemple, quatorze objets ne pèsent rien. Sur un journal réel, la mémoire occupée devient proportionnelle aux lignes valides, plus aux fenêtres de force brute. Le gain du générateur porte sur la lecture et sur le rejet anticipé des lignes inutiles, pas sur une analyse multi-règles en mémoire constante.

**La force brute est un heuristique de fenêtre, pas une preuve d'attaque.** Elle ne regarde que `LOGIN_FAILED` et `LOGIN_SUCCESS`, sans délai entre les événements : trois échecs espacés de plusieurs heures comptent comme trois échecs consécutifs. Le drapeau `deja_alertee` évite de répéter l'alerte à chaque échec supplémentaire, mais un seul succès suffit à réarmer. Le seuil est un paramètre, pas une constante cachée.

**L'audit exporte des types, pas des valeurs.** `globals()` et `locals()` sont bien appelés, et le rapport en est dérivé. Il ne dump pas le contenu des ensembles. C'est un choix de lisibilité. Un audit qui imprimerait `logs_valides` recopierait les données qu'il est censé seulement décrire.

**`router_payload` existe deux fois.** Celui de `parsing.py` distingue `TUPLE_AUTRE`. Celui de `pipeline.py` envoie ce cas dans `AUTRE`. `main.py` utilise le premier.

**Aucun composant réseau, aucun temps réel au sens d'un socket.** « Temps réel » est réalisé par la consommation au fil de l'eau d'un fichier ou d'un générateur. Le programme ne s'abonne pas à un flux syslog.


## 16. Conclusion

Le dépôt est un pipeline en quatre temps, tenu par des outils imposés plutôt que par une bibliothèque d'analyse de logs.

La lecture ne précède pas le traitement sous forme d'une grande liste : `yield` et `yield from` fournissent les lignes, et le parsing rejette au passage ce qui n'est pas un `LogRecord` exploitable. Les valeurs gardées sont des objets gelés, avec un tuple de connexion quand la ligne en porte un.

La transformation fonctionnelle (`filter`, `map`, `lambda`, `match`) est isolée dans `pipeline.py` et exécutée dans ce rapport. L'agrégation utilise un `set` d'IP en erreur, un `frozenset` de blacklist avec les quatre opérations, un `defaultdict` pour les comptes, et une détection de force brute dont l'alerte naît du `else` d'une boucle, uniquement quand la fenêtre d'échecs est homogène.

Le moteur de règles traite les contrôles comme des valeurs. `*args` reçoit les fonctions, `**kwargs` reçoit le seuil, la blacklist et les événements sensibles. L'audit photographie ensuite `globals()` et `locals()`, plus la portée de `traiter_les_logs`, pour laisser une trace des variables effectivement chargées à la fin du pipeline.

Sur les deux fichiers d'exemple, ce parcours garde 14 logs sur 17 lignes, signale alice et bob pour force brute, isole `203.0.113.66` comme seule IP à la fois active et blacklistée, et produit les constats `FIREWALL_BLOCK` et `DISK_ERROR`. Les tests automatiques du dépôt couvrent cette lecture.
